# Comparar referencias reservadas y examinar modelos publicados

**Pregunta:** ¿Qué cambia entre protocolos de referencia reservados y cómo responden los modelos publicados en un diagnóstico separado de artefactos fijos?

Nivel: 3. Requisitos previos: Niveles 1 y 2 de este tema. Tiempo de lectura: 50 minutos.



## Objetivos

La parte 1 ajusta cuatro referencias de recuperación con 64 filas y tres protocolos reservados. La parte 2 examina MLP/autoencoder/GP publicados en 32 estados de otra semilla, conserva omisiones sintéticas y mide cobertura centrada en GP. Las cohortes tienen exposición de entrenamiento diferente y nunca comparten un orden de modelos no vistos.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Importar las herramientas fijadas y resolver las entradas relativas a la carpeta del notebook.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
import hashlib
from pipeline import __version__ as engine_version
from pipeline.methods.screen import Screen
from pipeline.methods.learning import features
from pipeline.methods.observable import check_observable, PredictionRejected
from scipy.stats import norm

legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. Catálogo, constantes y contrato generan ambas cohortes simuladas. La parte 2 exige learning-v2 con versión/digest compatibles, exportaciones finales, escaladores JSON y bytes Cholesky en models/. Se imprimen SHA-256 completos, incluidos VERSION y registros de aprendizaje/contrato. No se incorporan etiquetas medidas ni datos descargados.

## Método

La parte 1 conserva cuatro ajustes con transformaciones solo de entrenamiento y reservas de 16/16/32 filas para interpolación, caso y grupo mineral. Su GP fijo aporta predicciones puntuales. La parte 2 usa build_design a semilla 20261101, ocho estados por caso; Screen.judge transforma inversamente tres objetivos neuronales y usa otra estandarización para reconstrucción. Pruebas de campos directos exceden límites globales de doce casos sin simular vectores inválidos. Screen.gp recibe variables estandarizadas como la red; media y desviación propias definen intervalos 50/80/95%, incluido ruido exportado. Cobertura y amplitud completas y por caso preceden selección opcional; subconjuntos vacíos tienen estimaciones nulas. La parte 2 no ajusta modelos ni umbrales ni usa ejecución Torch u ONNX.

## Comparación

**Parte 1: Reajustar cuatro referencias.** Los cuatro estimadores sklearn usan índices idénticos dentro de cada protocolo. El bosque usa cuarenta árboles; el refuerzo, cuarenta iteraciones y cuatro filas mínimas por hoja. El proceso gaussiano usa RBF fijo sin optimizar hiperparámetros y sin afirmar intervalos calibrados. Breiman (2001), Friedman (2001) y Rasmussen y Williams (2006) fundamentan las referencias. Se reservan dieciséis filas en interpolación y caso, y treinta y dos en grupo mineral. El pliegue de caso conserva el pórfido duro relacionado; el de grupo retira ambos casos.

Construir el diseño de aprendizaje desde las salidas físicas del circuito. Seleccionar dos casos relacionados de calcopirita, además de cobre oxidado y magnetita, para que la reserva de un grupo mineral retire juntos ambos ejemplos relacionados. Cada caso aporta dieciséis estados con semilla dentro de su contrato operativo. Son 64 etiquetas del simulador, no 64 muestras medidas de planta. Conservar este dominio pequeño al interpretar cada puntuación ajustada.

In [ ]:
# Construir el diseño de aprendizaje desde las salidas físicas del circuito.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.learning import build_design, interpolation_split, leave_one_case_out
from pipeline.methods.learning import leave_one_group_out, transfer_groups, FEATURES, TARGETS
import contextlib
import io

contract = json.loads(
    (repo / "data/derived/contract/operating_contract.json").read_text(encoding="utf-8")
)
case_ids = ("copper_porphyry_soft", "copper_porphyry_hard", "copper_oxide", "iron_magnetite_fine")
selected = tuple((CASE_BY_ID[name] for name in case_ids))
with contextlib.redirect_stdout(io.StringIO()):
    design = build_design(contract, cases=selected, per_case=16, seed=20261031)
x, y, case_of = (design["x"], design["y"], design["case"])
print("Filas del diseño:", len(x))
print("Variables:", len(FEATURES))
print("Objetivos:", TARGETS)


Inspeccionar la etiqueta nominal de `copper_porphyry_soft` antes de agrupar sus filas. Los porcentajes de recuperación comparten unidades, pero las propiedades minerales y los diagramas de proceso generan medias condicionales distintas. Leer la energía junto a la recuperación: un modelo puede reducir el residuo de una salida y fallar en otra, y una puntuación agregada no expresa ese compromiso.

In [ ]:
# Inspeccionar la etiqueta nominal de `copper_porphyry_soft` antes de agrupar sus filas.
case = CASE_BY_ID["copper_porphyry_soft"]
point = case.nominal
reference = simulate(case.ore, case.plant, point)
print("Caso de referencia:", case.id)
print("Recuperación de referencia (%):", round(reference.metrics["recovery_pct"], 3))
print("Energía de referencia (kWh/t):", round(reference.metrics["specific_energy_total_kwh_t"], 3))


Inspeccionar la etiqueta nominal de `iron_magnetite_fine` antes de agrupar sus filas. Los porcentajes de recuperación comparten unidades, pero las propiedades minerales y los diagramas de proceso generan medias condicionales distintas. Leer la energía junto a la recuperación: un modelo puede reducir el residuo de una salida y fallar en otra, y una puntuación agregada no expresa ese compromiso.

In [ ]:
# Inspeccionar la etiqueta nominal de `iron_magnetite_fine` antes de agrupar sus filas.
case = CASE_BY_ID["iron_magnetite_fine"]
point = case.nominal
reference = simulate(case.ore, case.plant, point)
print("Caso de referencia:", case.id)
print("Recuperación de referencia (%):", round(reference.metrics["recovery_pct"], 3))
print("Energía de referencia (kWh/t):", round(reference.metrics["specific_energy_total_kwh_t"], 3))


Construir tres preguntas de validación sobre exactamente el mismo diseño. La interpolación reserva un cuarto de cada caso. El pliegue por caso retira el pórfido blando y conserva su par duro. El pliegue por grupo mineral retira ambos casos de calcopirita. Presentar tamaños e identidades antes del ajuste permite detectar una fuga si un caso relacionado cruza accidentalmente la frontera del grupo. No ajustar hiperparámetros con filas reservadas.

In [ ]:
# Construir tres preguntas de validación sobre exactamente el mismo diseño.
train, test = interpolation_split(case_of, test_fraction=0.25, seed=20261031)
case_fold = next((f for f in leave_one_case_out(case_of) if f[0] == "copper_porphyry_soft"))
group_fold = leave_one_group_out(case_of, transfer_groups(selected))[0]
splits = {
    "Interpolación": (train, test),
    "Caso reservado": case_fold[1:],
    "Grupo mineral reservado": group_fold[1:],
}
for protocol, (training, testing) in splits.items():
    print(protocol, len(training), len(testing))
    print("Casos de entrenamiento:", sorted(set(case_of[training].tolist())))
    print("Casos de prueba:", sorted(set(case_of[testing].tolist())))
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

scores, predictions = ([], {})


Ridge penaliza los coeficientes al cuadrado después de escalar. Su respuesta lineal permite comprobar si la complejidad no lineal aporta algo con estas variables. El escalador se ajusta dentro de cada pliegue de entrenamiento, por lo que un mineral extremo reservado no modifica sus medias ni varianzas. Alpha permanece en uno, sin seleccionarlo mediante los errores de prueba mostrados.

In [ ]:
# Ridge penaliza los coeficientes al cuadrado después de escalar.
for protocol, (training, testing) in splits.items():
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    model.fit(x[training], y[training, 0])
    predicted = model.predict(x[testing])
    print("Rango de predicción (%):", np.round([predicted.min(), predicted.max()], 3))
    print("Rango de referencia (%):", np.round([y[testing, 0].min(), y[testing, 0].max()], 3))
    predictions["Ridge", protocol] = (testing, predicted)
    scores.append(
        {
            "Modelo": "Ridge",
            "Protocolo": protocol,
            "RMSE (pp)": float(np.sqrt(mean_squared_error(y[testing, 0], predicted))),
            "MAE (pp)": float(mean_absolute_error(y[testing, 0], predicted)),
            "R2": float(r2_score(y[testing, 0], predicted)),
        }
    )
display(pd.DataFrame(scores).tail(3).round(3))


El bosque aleatorio promedia cuarenta árboles con un mínimo de dos filas por hoja. Representa interacciones sin escalar variables, pero sus hojas predicen promedios de etiquetas de entrenamiento y no extrapolan una tendencia suave fuera de ellas. Comparar su error de transferencia con Ridge usando los mismos índices y etiquetas de recuperación. La cantidad fija de árboles acota la lección sin sustituir el algoritmo de ajuste.

In [ ]:
# El bosque aleatorio promedia cuarenta árboles con un mínimo de dos filas por hoja.
for protocol, (training, testing) in splits.items():
    model = RandomForestRegressor(
        n_estimators=40, min_samples_leaf=2, random_state=20261031, n_jobs=1
    )
    model.fit(x[training], y[training, 0])
    predicted = model.predict(x[testing])
    print("Rango de predicción (%):", np.round([predicted.min(), predicted.max()], 3))
    print("Rango de referencia (%):", np.round([y[testing, 0].min(), y[testing, 0].max()], 3))
    predictions["Bosque aleatorio", protocol] = (testing, predicted)
    scores.append(
        {
            "Modelo": "Bosque aleatorio",
            "Protocolo": protocol,
            "RMSE (pp)": float(np.sqrt(mean_squared_error(y[testing, 0], predicted))),
            "MAE (pp)": float(mean_absolute_error(y[testing, 0], predicted)),
            "R2": float(r2_score(y[testing, 0], predicted)),
        }
    )
display(pd.DataFrame(scores).tail(3).round(3))


El refuerzo por histogramas ajusta correcciones sucesivas al residuo de recuperación. Se declara un mínimo de cuatro filas por hoja porque el valor predeterminado impediría divisiones informativas en estos pliegues pequeños. Cuarenta iteraciones constituyen el presupuesto didáctico, no el ajuste de producción. Se desactiva la detención anticipada para que todos los pliegues sigan la misma regla explícita y ninguna fila de prueba determine cuándo parar.

In [ ]:
# El refuerzo por histogramas ajusta correcciones sucesivas al residuo de recuperación.
for protocol, (training, testing) in splits.items():
    model = HistGradientBoostingRegressor(
        max_iter=40, min_samples_leaf=4, early_stopping=False, random_state=20261031
    )
    model.fit(x[training], y[training, 0])
    predicted = model.predict(x[testing])
    print("Rango de predicción (%):", np.round([predicted.min(), predicted.max()], 3))
    print("Rango de referencia (%):", np.round([y[testing, 0].min(), y[testing, 0].max()], 3))
    predictions["Refuerzo por histogramas", protocol] = (testing, predicted)
    scores.append(
        {
            "Modelo": "Refuerzo por histogramas",
            "Protocolo": protocol,
            "RMSE (pp)": float(np.sqrt(mean_squared_error(y[testing, 0], predicted))),
            "MAE (pp)": float(mean_absolute_error(y[testing, 0], predicted)),
            "R2": float(r2_score(y[testing, 0], predicted)),
        }
    )
display(pd.DataFrame(scores).tail(3).round(3))


El proceso gaussiano usa un núcleo radial fijo después de estandarizar solo con entrenamiento. Desactivar la optimización de hiperparámetros evita advertencias de convergencia y hace reproducible la referencia acotada, aunque limita las conclusiones sobre el mejor proceso gaussiano alcanzable. Alpha regulariza las etiquetas del simulador. Esta corrida compara predicciones puntuales y no afirma que los intervalos predictivos estén calibrados.

In [ ]:
# El proceso gaussiano usa un núcleo radial fijo después de estandarizar solo con entrenamiento.
for protocol, (training, testing) in splits.items():
    model = make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(kernel=RBF(2.0), alpha=1.0, normalize_y=True, optimizer=None),
    )
    model.fit(x[training], y[training, 0])
    predicted = model.predict(x[testing])
    print("Rango de predicción (%):", np.round([predicted.min(), predicted.max()], 3))
    print("Rango de referencia (%):", np.round([y[testing, 0].min(), y[testing, 0].max()], 3))
    predictions["Proceso gaussiano", protocol] = (testing, predicted)
    scores.append(
        {
            "Modelo": "Proceso gaussiano",
            "Protocolo": protocol,
            "RMSE (pp)": float(np.sqrt(mean_squared_error(y[testing, 0], predicted))),
            "MAE (pp)": float(mean_absolute_error(y[testing, 0], predicted)),
            "R2": float(r2_score(y[testing, 0], predicted)),
        }
    )
display(pd.DataFrame(scores).tail(3).round(3))


Comparar el RMSE de recuperación dentro de cada pregunta de validación. Cada barra procede de un ajuste nuevo, no del archivo de producción.

In [ ]:
# Comparar el RMSE de recuperación dentro de cada pregunta de validación.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
score = pd.DataFrame(scores)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
score.pivot(index="Modelo", columns="Protocolo", values="RMSE (pp)").plot.barh(ax=ax)
ax.set(xlabel="RMSE de\nrecuperación\n(pp)", ylabel="", title="Error con diseño común")
ax.tick_params(axis="x", rotation=0)
handles, legend_names = ax.get_legend_handles_labels()
legend_short = {"Caso reservado": "Caso", "Grupo mineral reservado": "Grupo mineral"}
ax.legend(
    handles,
    [legend_short.get(name, name) for name in legend_names],
    fontsize=18,
    ncol=2,
    **legend_style,
)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)
print("Filas RMSE calculadas:", len(score))


Las barras separan interpolación de transferencia por caso y por grupo mineral. Comparar primero los estimadores dentro de un color: cada color tiene una población de entrenamiento distinta. Un error mayor al reservar el grupo evidencia dificultad con ese grupo ausente concreto. Cuatro casos de dieciséis filas no establecen un orden universal de estimadores, y un R² negativo se conserva como una falla frente a predecir la media de prueba.

Inspeccionar la estructura residual del grupo mineral ausente del entrenamiento. La diagonal representa el objetivo exacto del simulador.

In [ ]:
# Inspeccionar la estructura residual del grupo mineral ausente del entrenamiento.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 80})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for name in sorted({key[0] for key in predictions}):
    testing, predicted = predictions[name, "Grupo mineral reservado"]
    ax.scatter(y[testing, 0], predicted, label=name, s=20)
limits = [float(y[testing, 0].min()), float(y[testing, 0].max())]
ax.plot(limits, limits, "--", color="black")
ax.set(
    xlabel="Recuperación del motor\n(%)",
    ylabel="Recuperación\npredicha (%)",
    title="Predicciones del grupo ausente",
)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.61)


Los puntos alejados de la diagonal muestran el sentido del sesgo de transferencia que un RMSE único no revela. Una nube horizontal sugiere que el modelo vuelve hacia etiquetas conocidas cuando el mineral ausente ocupa otra región de variables. Ambos ejes usan recuperación, por lo que las distancias verticales son errores en puntos porcentuales. Las predicciones siguen aproximando las ecuaciones del circuito, sin compararse con recuperaciones medidas independientemente en flotación o separación magnética.

### Parte 2: Examinar los artefactos publicados

El primer experimento reajustó cuatro referencias con pliegues reservados. El segundo evalúa redes de producción ya entrenadas y otro GP de recuperación en 32 estados nuevos del simulador. Todas las familias de casos estuvieron disponibles durante entrenamiento de producción; otra semilla Sobol no convierte estos modelos en predictores de casos ausentes. No se ajustan pesos, escaladores, umbral ni parámetros GP. Una identidad de versión incompatible debe detener este experimento antes de inferencia.

Separar nombres de casos, unidades y rótulos de los identificadores de producción. Los dos experimentos comparten definiciones de variables y conservan cohortes e historiales diferentes.

In [ ]:
# Separar nombres de casos, unidades y rótulos de los identificadores de producción.
target_labels = ["Recuperación\n(%)", "Log10 de\nenriquecimiento", "Energía\n(kWh/t)"]
case_labels = {
    "all": "Los 32",
    "copper_porphyry_soft": "Blando",
    "copper_porphyry_hard": "Duro",
    "copper_oxide": "Óxido",
    "iron_magnetite_fine": "Magnetita",
}
probe_labels = ["Base", "Wi", "P80", "Agua"]
header_keys = (
    "cohort target unit attempted scored rejected rmse mae bias observable",
    "case probe row feature_value error threshold accepted subset level n hits coverage mean_width",
)
header_values = (
    "Cohorte|Objetivo|Unidad|Intentos|Puntuados|Rechazados|RMSE|MAE|Sesgo|Diagnósticos",
    "Caso|Prueba|Fila|Valor|Error|Umbral|Aceptados|Subconjunto",
    "Nivel nominal|N|Cubiertos|Cobertura|Amplitud media (pp)",
)
challenge_headers = dict(zip(" ".join(header_keys).split(), "|".join(header_values).split("|")))


def challenge_frame(rows):
    frame = pd.DataFrame(rows, dtype=object)
    labels = {
        "all": "Todas las filas",
        "guard_accepted": "Filas aceptadas por el guardia",
        "baseline": "Base",
    }
    for column in ("cohort", "subset", "probe"):
        if column in frame:
            frame[column] = frame[column].replace(labels)
    return frame.rename(columns=challenge_headers)


Definir la validación con las API importadas. Identidad, orden, umbral o bytes declarados incompatibles producen una falla clara antes de utilizar predicciones.

In [ ]:
# Definir la validación con las API importadas.
def released_provenance(record, contract, version, model_dir):
    identity = (record.get("schema"), record.get("engine_version"), record.get("contract_digest"))
    if identity != ("oreflow.learning/v2", version, contract["digest"]) or record.get("reused"):
        raise ValueError("Identidad de publicación incompatible: " + str(identity))
    scalers = json.loads((model_dir / "process_surrogate.json").read_text(encoding="utf-8"))
    for key, expected in [("features", list(FEATURES)), ("targets", list(TARGETS))]:
        if record[key] != expected or scalers[key] != expected:
            raise ValueError("Orden incompatible de variables u objetivos: " + key)
    if scalers["guard_threshold"] != record["final"]["guard_threshold"]:
        raise ValueError("Umbral publicado incompatible")
    exports = record["final"]["exports"]
    names = [("surrogate", "process_surrogate.onnx"), ("guard", "process_guard.onnx")]
    declared = [(exports[key], name) for key, name in names]
    declared += [(exports["screen"], "process_screen.json")]
    declared += [(exports["screen"]["cholesky"], "process_gp_cholesky.bin")]
    hashes = {}
    for entry, name in declared:
        raw = (model_dir / name).read_bytes()
        digest = hashlib.sha256(raw).hexdigest()
        if (
            entry["path"] != name
            or len(raw) != entry["bytes"]
            or entry.get("sha256", digest) != digest
        ):
            raise ValueError("Identidad de bytes del modelo incompatible: " + name)
        hashes[name] = digest
    hashes["process_surrogate.json"] = hashlib.sha256(
        (model_dir / "process_surrogate.json").read_bytes()
    ).hexdigest()
    return hashes


Verificar la publicación adoptada e imprimir procedencia. Recalcular el digest desde JSON canónico, conservar la huella del productor y mostrar SHA-256 completos. Artefactos ausentes o antiguos representan una frontera real de datos, sin sustituirlos por ajustes de referencia. Los archivos ONNX solo aportan bytes para identificación; la inferencia usa Screen con NumPy/SciPy.

In [ ]:
# Verificar la publicación adoptada e imprimir procedencia.
model_dir = repo / "models"
release_path = repo / "data/derived/learning.json"
released = json.loads(release_path.read_text(encoding="utf-8"))
artifact_hashes = released_provenance(released, contract, engine_version, model_dir)
contract_body = {key: value for key, value in contract.items() if key != "digest"}
contract_bytes = json.dumps(contract_body, sort_keys=True, separators=(",", ":")).encode()
if hashlib.sha256(contract_bytes).hexdigest() != contract["digest"]:
    raise ValueError("Digest contractual incompatible")
for label, path in [
    ("learning.json", release_path),
    ("VERSION", repo / "VERSION"),
    ("operating_contract.json", repo / "data/derived/contract/operating_contract.json"),
]:
    artifact_hashes[label] = hashlib.sha256(path.read_bytes()).hexdigest()
print("Versión publicada:", engine_version)
print("Digest contractual:", contract["digest"])
print("Huella del productor:", released["execution"]["fingerprint"])
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(artifact_hashes.items(), columns=["Artefacto", "SHA-256"]))
frozen = Screen(model_dir)
scalers = json.loads((model_dir / "process_surrogate.json").read_text(encoding="utf-8"))
for scale in (frozen.feature_scale, frozen.target_scale, frozen.guard_scale):
    if not np.isfinite(scale).all() or not (scale > 0).all():
        raise ValueError("Escalado publicado inválido")
frozen_threshold = frozen.guard_threshold
print("Umbral fijo del guardia:", frozen_threshold)


La tolerancia de exportación se declara en unidades neuronales estandarizadas. Dividir las diferencias físicas por la escala publicada de cada objetivo, con tolerancia relativa cero y tolerancia absoluta declarada sin cambios. Formas inválidas, valores no finitos y escalas no positivas producen fallas.

In [ ]:
# La tolerancia de exportación se declara en unidades neuronales estandarizadas.
def released_parity(actual, expected, scale, tolerance):
    actual, expected, scale = (np.asarray(actual), np.asarray(expected), np.asarray(scale))
    if actual.shape != expected.shape or scale.shape != actual.shape:
        raise ValueError("Forma incompatible de paridad")
    if (
        not all((np.isfinite(value).all() for value in (actual, expected, scale)))
        or not (scale > 0).all()
    ):
        raise ValueError("La paridad exige valores finitos y escalas positivas")
    if not np.isfinite(tolerance) or tolerance < 0:
        raise ValueError("Tolerancia estandarizada inválida")
    return bool(np.allclose((actual - expected) / scale, 0, rtol=0, atol=tolerance))


for row in scalers["reference"]:
    observed = frozen.judge(row["features"])["prediction"]
    expected = np.array([row["prediction"][target] for target in TARGETS])
    actual = np.array([observed[target] for target in TARGETS])
    if not released_parity(
        actual, expected, frozen.target_scale, float(released["settings"]["onnx_tolerance"])
    ):
        raise ValueError("Referencia de red/escalador incompatible: " + row["case_id"])


Inspeccionar las coordenadas exportadas. La red usa (x-feature_mean)/feature_scale y transforma cada salida mediante target_scale y target_mean. El autoencoder utiliza medias y escalas propias y error cuadrático medio de reconstrucción. El GP recibe variables estandarizadas como la red y devuelve media y desviación de recuperación en unidades porcentuales. Los transformadores permanecen fijos; esta red pequeña no reproduce el entrenamiento profundo de Hinton y Salakhutdinov.

In [ ]:
# Inspeccionar las coordenadas exportadas.
scale_table = pd.DataFrame(
    {
        "Variable": FEATURES,
        "Media de red": frozen.feature_mean,
        "Escala de red": frozen.feature_scale,
        "Media del guardia": frozen.guard_mean,
        "Escala del guardia": frozen.guard_scale,
    }
)
display(scale_table)
display(
    pd.DataFrame(
        {
            "Objetivo": TARGETS,
            "Media de objetivo": frozen.target_mean,
            "Escala de objetivo": frozen.target_scale,
        }
    )
)
example_raw = np.asarray(scalers["reference"][0]["features"])
example_scaled = (example_raw - frozen.feature_mean) / frozen.feature_scale
example_targets = (
    frozen.forward("surrogate", example_scaled) * frozen.target_scale + frozen.target_mean
)
display(pd.Series(example_targets, index=TARGETS, name="Predicción en unidades físicas"))


Verificar el mapeo de variables continuas directas frente a OperatingPoint en cada caso. Los límites globales incluyen los doce casos de entrenamiento, no solo los cuatro examinados. work_index_kwh_t, target_p80_um y water_m3_t corresponden a campos directos; se excluyen razones transformadas. Así la afirmación de exterioridad se apoya en límites contractuales y mapeo explícito, sin inferirla de coordenadas estandarizadas ni extremos observados parciales.

In [ ]:
# Verificar el mapeo de variables continuas directas frente a OperatingPoint en cada caso.
probe_features = ("work_index_kwh_t", "target_p80_um", "water_m3_t")
if set(released["design"]["cases"]) != set(contract["cases"]) or len(contract["cases"]) != 12:
    raise ValueError("Inventario global de casos incompatible")
global_bounds = {}
for name in probe_features:
    entries = [item["inputs"][name] for item in contract["cases"].values()]
    global_bounds[name] = (
        min((item["min"] for item in entries)),
        max((item["max"] for item in entries)),
    )
for case_id in contract["cases"]:
    member = CASE_BY_ID[case_id]
    mapped = features(member, member.nominal, {"liberation_size": 1.0, "floatability": 1.0})
    for name in probe_features:
        if mapped[FEATURES.index(name)] != getattr(member.nominal, name):
            raise ValueError("Mapeo directo de variable incompatible: " + name)
units = {item["name"]: item["unit"] for item in contract["inputs"]}
bounds_table = pd.DataFrame.from_dict(
    global_bounds, orient="index", columns=["Mínimo global", "Máximo global"]
)
bounds_table["Índice"] = [FEATURES.index(name) for name in bounds_table.index]
bounds_table["Unidad"] = [units[name] for name in bounds_table.index]
display(bounds_table.rename_axis("Variable"))


Construir otro diseño con ocho estados válidos por cada uno de los cuatro casos: 32 estados a semilla 20261101. build_design calcula variables y los tres engine.targets mediante resoluciones nuevas. La semilla se declara antes de observar errores y difiere de la referencia y del entrenamiento publicado. Estos sorteos de familias conocidas examinan artefactos fijos; no constituyen reajuste LOCO ni transferencia medida en planta.

In [ ]:
# Construir otro diseño con ocho estados válidos por cada uno de los cuatro casos: 32 estados a
# semilla 20261101.
challenge_seed = 20261101
if challenge_seed in (20261031, released["design"]["seed"]):
    raise ValueError("La semilla debe declararse por separado")
with contextlib.redirect_stdout(io.StringIO()):
    challenge = build_design(contract, cases=selected, per_case=8, seed=challenge_seed)
challenge_x, challenge_y, challenge_case = (challenge["x"], challenge["y"], challenge["case"])
if challenge_x.shape != (32, len(FEATURES)) or challenge_y.shape != (32, len(TARGETS)):
    raise ValueError("Forma incompatible del diseño de diagnóstico")
display(pd.Series(challenge_case).value_counts().rename("Filas del diagnóstico"))
print("Semilla del diagnóstico:", challenge_seed)
print("Orden de objetivos:", TARGETS)


Definir inferencia con los tres objetivos en su orden declarado. El orden de inserción de un diccionario no puede intercambiar recuperación, log-upgrade y energía. Valores no finitos o declaraciones incompatibles detienen el experimento. La identidad del caso no integra las variables y ningún artefacto se entrena con estas filas.

In [ ]:
# Definir inferencia con los tres objetivos en su orden declarado.
def released_predictions(screen, raw_x):
    if tuple(screen.targets) != tuple(TARGETS):
        raise ValueError("Orden publicado incompatible")
    rows = [screen.judge(row)["prediction"] for row in raw_x]
    predicted = np.array([[row[name] for name in TARGETS] for row in rows])
    if predicted.shape != (len(raw_x), len(TARGETS)) or not np.isfinite(predicted).all():
        raise ValueError("Predicciones inválidas de los tres objetivos")
    return predicted


challenge_prediction = released_predictions(frozen, challenge_x)
target_units = ("pp", "razón log10", "kWh/t")
display(
    pd.DataFrame(
        {
            "Objetivo": TARGETS,
            "Unidad": target_units,
            "Mínimo predicho": challenge_prediction.min(axis=0),
            "Máximo predicho": challenge_prediction.max(axis=0),
            "Mínimo de referencia": challenge_y.min(axis=0),
            "Máximo de referencia": challenge_y.max(axis=0),
        }
    )
)


Usar reference_range conservado por ridge en la publicación compatible: mínimos y máximos del diseño completo de producción. Son límites de un diseño finito, no límites físicos exhaustivos. La misma referencia es obligatoria para puntuación agrupada y por caso; cada subconjunto conserva rangos observados y predichos propios. Esta comprobación amplia posterior a inferencia no ajusta, escala, selecciona ni recorta predicciones y no establece validez física fila por fila.

In [ ]:
# Usar reference_range conservado por ridge en la publicación compatible: mínimos y máximos del
# diseño completo de producción.
def released_reference(record):
    scores = record["interpolation"]["models"]["ridge"]
    bounds = np.asarray(
        [scores[target]["observable"]["reference_range"] for target in TARGETS], dtype=float
    )
    if (
        bounds.shape != (len(TARGETS), 2)
        or not np.isfinite(bounds).all()
        or (bounds[:, 1] < bounds[:, 0]).any()
    ):
        raise ValueError("Límites inválidos de referencia completa")
    return bounds.T


authored_reference = released_reference(released)
print(
    "Referencia: interpolation/ridge observable.reference_range publicado; filas del diseño completo:",
    released["design"]["rows"],
)
display(
    pd.DataFrame(
        authored_reference.T,
        index=TARGETS,
        columns=["Mínimo del diseño completo", "Máximo del diseño completo"],
    )
)


Calcular errores por objetivo primero en las 32 filas y después por caso. Las unidades difieren; no se promedian objetivos ni se incorpora esta red al orden de las cuatro referencias. La comprobación de escala conserva predicciones finitas deficientes; una cohorte totalmente fuera de escala mantiene rangos reales y errores nulos, con denominadores de intentos, puntuados y rechazados. Errores de forma o valores no finitos detienen la ejecución.

In [ ]:
# Calcular errores por objetivo primero en las 32 filas y después por caso.
def challenge_errors(observed, predicted, cohort, reference):
    rows = []
    for j, target in enumerate(TARGETS):
        error = predicted[:, j] - observed[:, j]
        rmse, mae, bias = (None, None, None)
        try:
            diagnostic = check_observable(
                observed[:, j], predicted[:, j], reference=reference[:, j]
            )
            rmse, mae, bias = (
                float(np.sqrt(np.mean(error**2))),
                float(np.mean(abs(error))),
                float(error.mean()),
            )
        except PredictionRejected as rejection:
            diagnostic = rejection.diagnostics
        valid = rmse is not None
        row = dict(cohort=cohort, target=target, unit=target_units[j], attempted=len(error))
        row.update(scored=len(error) if valid else 0, rejected=0 if valid else len(error))
        row.update(rmse=rmse, mae=mae, bias=bias, observable=diagnostic)
        rows.append(row)
    return rows


neural_errors = challenge_errors(challenge_y, challenge_prediction, "all", authored_reference)
for case_id in case_ids:
    mask = challenge_case == case_id
    neural_errors += challenge_errors(
        challenge_y[mask], challenge_prediction[mask], case_id, authored_reference
    )
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(challenge_frame(neural_errors))


Graficar los tres objetivos de la red fija por separado en sus coordenadas físicas. Cada diagonal representa identidad con el motor, sin ajustar una línea de calibración.

In [ ]:
# Graficar los tres objetivos de la red fija por separado en sus coordenadas físicas.
plt.rcParams.update({"font.size": 20, "axes.titlepad": 8})
for indices in ((0,), (1, 2)):
    fig, axes = plt.subplots(1, len(indices), figsize=(6.4, 3.8), squeeze=False)
    for axis, j in zip(axes[0], indices):
        for case_id in case_ids:
            mask = challenge_case == case_id
            axis.scatter(
                challenge_y[mask, j],
                challenge_prediction[mask, j],
                label=case_labels[case_id],
                s=24,
            )
        limits = [float(challenge_y[:, j].min()), float(challenge_y[:, j].max())]
        axis.plot(limits, limits, "--", color="black")
        axis.set_title(target_labels[j], fontsize=20)
        axis.locator_params(nbins=3)
    fig.supxlabel("Objetivo del motor", fontsize=20)
    fig.supylabel("Predicción neuronal", fontsize=20)
    handles, names = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, names, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=20)
    fig.subplots_adjust(left=0.18, right=0.98, bottom=0.2, top=0.54, wspace=0.7)


Cada panel tiene unidades propias: separación vertical en puntos porcentuales de recuperación, log10 de enriquecimiento o energía específica. Todas las filas permanecen visibles con independencia del guardia. La tabla conserva rangos aunque se rechace una cohorte por escala. El modelo final conoció todas las familias; no puede añadirse como otro estimador reservado al primer experimento ni establecer capacidad predictiva en una planta operativa.

Definir el diagnóstico de reconstrucción para bases sin cambios y desplazamientos controlados. Cada prueba sustituye una coordenada directa, alternando por debajo o encima del intervalo global con un margen completo. Estos vectores pueden romper relaciones entre variables: son diagnósticos sintéticos, nunca estados de planta ni entradas a simulate. Una prueba desplazada aceptada permanece como detección omitida y no se elimina del denominador.

In [ ]:
# Definir el diagnóstico de reconstrucción para bases sin cambios y desplazamientos controlados.
def challenge_guard_row(screen, raw, case_id, probe, index, value):
    error = float(screen.judge(raw)["guard_error"])
    threshold = screen.guard_threshold
    return {
        "case": case_id,
        "probe": probe,
        "row": index,
        "feature_value": value,
        "error": error,
        "threshold": threshold,
        "accepted": error <= threshold,
    }


baseline_diagnostics = [
    challenge_guard_row(frozen, row, challenge_case[i], "baseline", i, None)
    for i, row in enumerate(challenge_x)
]
display(challenge_frame(baseline_diagnostics))


Construir el diagnóstico completo: bases sin cambios y desplazamientos de una coordenada fuera de cada límite global. Cada desplazamiento aceptado permanece en las filas devueltas.

In [ ]:
# Construir el diagnóstico completo: bases sin cambios y desplazamientos de una coordenada fuera de
# cada límite global.
def challenge_guard(screen, raw_x, case_of, bounds, baselines=None):
    rows = (
        list(baselines)
        if baselines is not None
        else [
            challenge_guard_row(screen, row, case_of[i], "baseline", i, None)
            for i, row in enumerate(raw_x)
        ]
    )
    threshold = screen.guard_threshold
    for i, raw in enumerate(raw_x):
        for name, (low, high) in bounds.items():
            probe = raw.copy()
            span = max(1.0, abs(low), abs(high), high - low)
            value = low - span if i % 2 == 0 else high + span
            if not (value < low or value > high):
                raise ValueError("La prueba sintética debe superar límites globales")
            probe[FEATURES.index(name)] = value
            rows.append(challenge_guard_row(screen, probe, case_of[i], name, i, value))
    if not np.isfinite([row["error"] for row in rows]).all() or screen.guard_threshold != threshold:
        raise ValueError("Error no finito o umbral modificado")
    return rows


guard_rows = challenge_guard(
    frozen, challenge_x, challenge_case, global_bounds, baseline_diagnostics
)
guard_table = pd.DataFrame(guard_rows)
display(challenge_frame(guard_rows))


Conservar cantidades aceptadas y rechazadas por tipo de prueba y caso, incluso si un guardia insensible acepta todos los desplazamientos. Rechazos de base describen banderas sobre estados válidos del simulador; desplazamientos aceptados describen omisiones en este diagnóstico construido. Ninguno estima frecuencia de fallas de planta. Se examina solo el autoencoder, sin la regla adicional de semiamplitud GP utilizada por el optimizador.

In [ ]:
# Conservar cantidades aceptadas y rechazadas por tipo de prueba y caso, incluso si un guardia
# insensible acepta todos los desplazamientos.
guard_counts = guard_table.groupby(["probe", "case"], sort=False)["accepted"].agg(["size", "sum"])
guard_counts.columns = ["attempted", "accepted"]
guard_counts["rejected"] = guard_counts["attempted"] - guard_counts["accepted"]
display(challenge_frame(guard_counts.reset_index()))
baseline_guard = guard_table[guard_table["probe"] == "baseline"]
shifted_guard = guard_table[guard_table["probe"] != "baseline"]
guard_accepted = baseline_guard["accepted"].to_numpy(dtype=bool)
print("Bases aceptadas/rechazadas:", int(guard_accepted.sum()), int((~guard_accepted).sum()))
print(
    "Desplazadas aceptadas/rechazadas:",
    int(shifted_guard["accepted"].sum()),
    int((~shifted_guard["accepted"]).sum()),
)
if frozen.guard_threshold != frozen_threshold:
    raise ValueError("Umbral fijo modificado")


Comparar errores con el mismo umbral fijo. Dividir el eje vertical por ese umbral sitúa la frontera en uno; symlog conserva errores cercanos a cero y pruebas extremas.

In [ ]:
# Comparar errores con el mismo umbral fijo.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 80})
with plt.rc_context({"font.size": 18}):
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    if frozen_threshold <= 0:
        raise ValueError("Se exige umbral positivo para graficar razones")
    for index, name in enumerate(("baseline",) + probe_features):
        rows = guard_table[guard_table["probe"] == name]
        ax.scatter(np.full(len(rows), index), rows["error"] / frozen_threshold, s=24)
    ax.axhline(1, linestyle="--", color="black", label="Umbral fijo")
    ax.set_xticks(range(4), probe_labels, rotation=18)
    ax.set_yscale("symlog", linthresh=1)
    ax.set(ylabel="Error de\nreconstrucción /\numbral", title="Diagnóstico controlado de variables")
    ax.legend(fontsize=18, ncol=2, **legend_style)
    fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.61)


Puntos en uno o por debajo se aceptan, incluidos vectores desplazados no detectados. Puntos por encima se rechazan, incluidas posibles bases sin cambios. La sustitución prueba exterioridad de una coordenada frente a todos los límites declarados de entrenamiento, sin aislar una intervención física realizable. El umbral permanece publicado; ajustarlo después de observar la figura cambiaría la pregunta e invalidaría la prueba fija.

Definir resúmenes gaussianos a niveles nominales 0.5, 0.8 y 0.95. En una cohorte no vacía, cobertura es la proporción de recuperaciones exactas dentro de media GP ± z por desviación GP; la amplitud media usa las mismas filas. Subconjuntos vacíos carecen de cantidad cubierta, cobertura y amplitud estimadas. Ningún intervalo se centra sobre una predicción neuronal.

In [ ]:
# Definir resúmenes gaussianos a niveles nominales 0.5, 0.8 y 0.95.
def challenge_coverage_rows(mean, std, truth, mask, cohort, subset):
    rows = []
    n = int(np.count_nonzero(mask))
    for level in (0.5, 0.8, 0.95):
        half = norm.ppf((1 + level) / 2) * std[mask]
        hits = int(np.count_nonzero(abs(truth[mask] - mean[mask]) <= half)) if n else None
        rows.append(
            {
                "subset": subset,
                "cohort": cohort,
                "level": level,
                "n": n,
                "hits": hits,
                "coverage": hits / n if n else None,
                "mean_width": float(np.mean(2 * half)) if n else None,
            }
        )
    return rows


Obtener media y desviación desde Screen.gp sobre entradas estandarizadas. Formar cohortes completas y por caso antes de subconjuntos aceptados, conservando estimaciones nulas si están vacíos.

Evaluar el GP de producción en cada estado antes de seleccionar. Mostrar primero cohorte completa y casos; después el subconjunto opcional aceptado por el guardia. Ocho filas por caso producen incrementos de cobertura de 0.125, por lo que el acuerdo nominal es descriptivo y grueso. El GP incluye el término de ruido exportado. No se reproduce calibración LOCO por pliegues ni se estima cobertura frente a una planta medida independientemente.

In [ ]:
# Obtener media y desviación desde Screen.gp sobre entradas estandarizadas.
def challenge_coverage(screen, raw_x, truth, case_of, accepted):
    standardized = (raw_x - screen.feature_mean) / screen.feature_scale
    gp_values = np.array([screen.gp(row) for row in standardized])
    if not np.isfinite(gp_values).all() or (gp_values[:, 1] < 0).any():
        raise ValueError("Media o desviación GP inválidas")
    mean, std = (gp_values[:, 0], gp_values[:, 1])
    cohorts = [("all", np.ones(len(raw_x), dtype=bool))]
    cohorts += [(name, case_of == name) for name in dict.fromkeys(case_of.tolist())]
    rows = []
    for subset, selected_rows in [
        ("all", np.ones(len(raw_x), dtype=bool)),
        ("guard_accepted", accepted),
    ]:
        for cohort, mask in cohorts:
            rows += challenge_coverage_rows(mean, std, truth, mask & selected_rows, cohort, subset)
    return (rows, mean, std)


coverage_rows, released_gp_mean, released_gp_std = challenge_coverage(
    frozen, challenge_x, challenge_y[:, 0], challenge_case, guard_accepted
)
coverage_all = [row for row in coverage_rows if row["subset"] == "all"]
coverage_accepted = [row for row in coverage_rows if row["subset"] == "guard_accepted"]
display(challenge_frame(coverage_all))
display(challenge_frame(coverage_accepted))
gp95 = next((row for row in coverage_all if row["cohort"] == "all" and row["level"] == 0.95))
print("Centro del intervalo GP: media del GP de producción")
print("Resolución de cobertura por caso:", 1 / 8)


Graficar solo curvas de cobertura sin filtrar, conservando cada fila. La diagonal representa cobertura nominal, sin ajustar una corrección de calibración.

In [ ]:
# Graficar solo curvas de cobertura sin filtrar, conservando cada fila.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 80})
with plt.rc_context({"font.size": 18}):
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    for cohort in ("all",) + case_ids:
        records = [row for row in coverage_all if row["cohort"] == cohort]
        ax.plot(
            [row["level"] for row in records],
            [row["coverage"] for row in records],
            marker="o",
            label=case_labels[cohort],
        )
    ax.plot([0, 1], [0, 1], "--", color="black")
    ax.set(
        xlim=(0, 1),
        ylim=(0, 1.02),
        xlabel="Cobertura nominal",
        ylabel="Cobertura\nempírica",
        title="Cobertura GP sin filtrar",
    )
    ax.legend(fontsize=18, ncol=3, **legend_style)
    fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.61)


Cobertura bajo la diagonal identifica cobertura insuficiente en estos sorteos; estar por encima no establece incertidumbre útil, porque intervalos muy amplios pueden cubrir casi todo. Amplitudes medias y denominadores de la tabla califican las curvas. Incrementos de un octavo por caso y solo cuatro casos conocidos limitan el diagnóstico. Filtrar después de observar resultados ocultaría fallas; las cifras del subconjunto aceptado permanecen separadas.

Mostrar intervalos GP de producción al 95 por ciento en las 32 filas. El centro es la media predictiva propia del GP; marcas del motor muestran omisiones individuales sin sustituirlas por predicciones de la red neuronal.

In [ ]:
# Mostrar intervalos GP de producción al 95 por ciento en las 32 filas.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 80})
with plt.rc_context({"font.size": 18}):
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    rows = np.arange(len(challenge_x))
    half95 = norm.ppf(0.975) * released_gp_std
    ax.errorbar(rows, released_gp_mean, yerr=half95, fmt="o", markersize=4, label="Media GP ±95%")
    ax.scatter(rows, challenge_y[:, 0], marker="x", color="black", label="Motor")
    ax.set(xlabel="Fila del diagnóstico", ylabel="Recuperación (%)", title="Intervalos GP y motor")
    ax.legend(fontsize=18, ncol=2, **legend_style)
    fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.61)


Una marca del motor fuera del segmento vertical indica omisión, aunque el autoencoder acepte ese vector. Red y GP tienen parámetros y objetivos ajustados diferentes; utilizar centro neuronal produciría un intervalo que el GP nunca predijo. Tabla y figura usan las 32 etiquetas antes de filtrar. Ninguna probabilidad gaussiana nominal certifica incertidumbre calibrada de planta ni transferencia a un grupo mineral ausente.

Vincular el diagnóstico publicado por separado de las cuatro referencias. Los rangos conservan tres objetivos, las cantidades del guardia conservan omisiones y el GP sin filtrar mantiene su denominador. Errores nulos no se sustituyen por cero. Estos resúmenes describen los archivos identificados y deben recalcularse después de adoptar la publicación.

In [ ]:
# Vincular el diagnóstico publicado por separado de las cuatro referencias.
released_values = {
    "Filas del diagnóstico": len(challenge_x),
    "Mínimo neuronal de recuperación": challenge_prediction[:, 0].min(),
    "Máximo neuronal de recuperación": challenge_prediction[:, 0].max(),
    "Mínimo neuronal de log-enriquecimiento": challenge_prediction[:, 1].min(),
    "Máximo neuronal de log-enriquecimiento": challenge_prediction[:, 1].max(),
    "Mínimo neuronal de energía": challenge_prediction[:, 2].min(),
    "Máximo neuronal de energía": challenge_prediction[:, 2].max(),
    "Cohortes de objetivo puntuadas": sum((row["rmse"] is not None for row in neural_errors)),
    "Cohortes de objetivo rechazadas": sum((row["rmse"] is None for row in neural_errors)),
    "Bases aceptadas": guard_accepted.sum(),
    "Bases rechazadas": (~guard_accepted).sum(),
    "Pruebas sintéticas": len(shifted_guard),
    "Sintéticas aceptadas": shifted_guard["accepted"].sum(),
    "Sintéticas rechazadas": (~shifted_guard["accepted"]).sum(),
    "Umbral publicado del guardia": frozen_threshold,
    "Cobertura GP95 publicada": gp95["coverage"],
    "Amplitud GP95 publicada": gp95["mean_width"],
}
print(
    json.dumps(
        {label: round(float(value), 6) for label, value in released_values.items()}, allow_nan=False
    )
)


## Resultados

**Respuesta:** El proceso gaussiano fijo tiene errores de caso/grupo <!-- result:RMSE de caso del proceso gaussiano (pp) -->0<!-- /result -->/<!-- result:RMSE de grupo del proceso gaussiano (pp) -->0<!-- /result --> pp y R² de grupo <!-- result:R2 de grupo del proceso gaussiano -->0<!-- /result -->. <!-- result:Modelos con R2 de grupo negativo -->0<!-- /result --> estimadores tienen R² de grupo negativo. Las figuras conservan fallas de transferencia y comparan protocolos con filas comunes y transformaciones solo de entrenamiento; ninguno establece predicción sobre una planta medida.

**Diagnóstico separado de artefactos publicados:** <!-- result:Filas del diagnóstico -->0<!-- /result --> estados simulados producen rangos neuronales de <!-- result:Mínimo neuronal de recuperación -->0<!-- /result --> a <!-- result:Máximo neuronal de recuperación -->0<!-- /result -->% de recuperación, <!-- result:Mínimo neuronal de log-enriquecimiento -->0<!-- /result --> a <!-- result:Máximo neuronal de log-enriquecimiento -->0<!-- /result --> log10 de enriquecimiento y <!-- result:Mínimo neuronal de energía -->0<!-- /result --> a <!-- result:Máximo neuronal de energía -->0<!-- /result --> kWh/t. La tabla conserva <!-- result:Cohortes de objetivo puntuadas -->0<!-- /result --> filas de objetivo/cohorte puntuadas y <!-- result:Cohortes de objetivo rechazadas -->0<!-- /result --> rechazadas, con errores por unidad y rangos reales. Al umbral fijo <!-- result:Umbral publicado del guardia -->0<!-- /result -->, las bases aceptadas/rechazadas son <!-- result:Bases aceptadas -->0<!-- /result -->/<!-- result:Bases rechazadas -->0<!-- /result -->. De <!-- result:Pruebas sintéticas -->0<!-- /result --> vectores fuera de límites, <!-- result:Sintéticas aceptadas -->0<!-- /result --> son omisiones aceptadas y <!-- result:Sintéticas rechazadas -->0<!-- /result --> se rechazan. Los intervalos al 95% centrados en GP cubren fracción <!-- result:Cobertura GP95 publicada -->0<!-- /result --> de todas las recuperaciones, con amplitud media <!-- result:Amplitud GP95 publicada -->0<!-- /result --> pp. N=8 por caso es grueso. Este diagnóstico fijo permanece separado del orden reservado y no reproduce calibración LOCO ni validez en planta.

Resumir errores de caso y grupo y conteos R2 negativos desde los cuatro ajustes de referencia reales.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
score_by_key = {(row["Modelo"], row["Protocolo"]): row for row in scores}
answer_values = {}
answer_values.update(
    {
        "RMSE de caso de Ridge (pp)": score_by_key["Ridge", "Caso reservado"]["RMSE (pp)"],
        "RMSE de caso del bosque (pp)": score_by_key["Bosque aleatorio", "Caso reservado"][
            "RMSE (pp)"
        ],
    }
)
answer_values.update(
    {
        "RMSE de caso del proceso gaussiano (pp)": score_by_key[
            "Proceso gaussiano", "Caso reservado"
        ]["RMSE (pp)"],
        "RMSE de grupo del proceso gaussiano (pp)": score_by_key[
            "Proceso gaussiano", "Grupo mineral reservado"
        ]["RMSE (pp)"],
        "R2 de grupo del proceso gaussiano": score_by_key[
            "Proceso gaussiano", "Grupo mineral reservado"
        ]["R2"],
        "Modelos con R2 de grupo negativo": sum(
            (row["R2"] < 0 for row in scores if row["Protocolo"] == "Grupo mineral reservado")
        ),
    }
)
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

El orden de referencias reservadas y el diagnóstico de artefactos fijos responden preguntas diferentes. Se conservan errores de tres objetivos, omisiones sintéticas y cobertura GP sin filtrar aunque sean desfavorables. El modelo publicado conoció todas las familias; 32 filas no reproducen calibración LOCO. N=8 por caso aporta descripción gruesa, sin validez calibrada en planta. La reconstrucción usa una red pequeña, sin afirmar reproducción de la arquitectura profunda de Hinton y Salakhutdinov.

## Referencias

- https://doi.org/10.1023/A:1010933404324
- https://doi.org/10.1214/aos/1013203451
- https://doi.org/10.1017/CBO9780511790423
- https://doi.org/10.1023/A:1010933404324
- Hinton y Salakhutdinov (2006), reconstrucción neuronal. https://doi.org/10.1126/science.1127647
- Material complementario de los autores. https://www.cs.toronto.edu/~rsalakhu/papers/science_som.pdf
- Rasmussen y Williams (2006), Gaussian Processes for Machine Learning. https://gaussianprocess.org/gpml/
- Media y varianza predictivas, capítulo 2, algoritmo 2.1. https://gaussianprocess.org/gpml/chapters/
